# 02 — shock propagation on the network

SIR-style simulation: inject a shock at one asset, watch it spread along the directed edges.
then the key check — do the *simulated* travel times agree with *empirical* lead-lag measured straight from returns?

In [ ]:
from diffuse import data as D
from diffuse import network as N
from diffuse import diffusion as DI

prices = D.fetch_prices(D.UNIVERSE[:10], interval="1h", period="1mo")
rets = D.to_returns(prices)
G = N.build_network(rets, n_shuffles=100, verbose=False)
leaders = N.top_leaders(G, k=3)
print("seeding shocks at:", leaders)

In [ ]:
# how long does a shock at the top leader take to reach everyone else?
times = DI.simulate_shock(G, leaders[0], n_sims=100)
for node, t in sorted(times.items(), key=lambda kv: (kv[1] is None, kv[1]))[:8]:
    print(node, round(float(t), 1) if t == t else "never reached")

In [ ]:
rho, p, n = DI.validate_against_leadlag(G, rets, n_sims=100)
print("spearman(sim travel time, empirical lead-lag) = %.3f, p = %.3f (n=%d pairs)" % (rho, p, n))

## reading this

- positive rho means pairs that are near each other in the simulated cascade also show short empirical lead-lag. that is the network earning its keep.
- rho near zero (or negative) means the diffusion story does not match the data — the network might still have predictive edges, but the SIR metaphor is not the right mental model. say so honestly.
- on hourly ETF data I would expect weak-to-moderate agreement: the market factor dominates everything, so most travel times are short and noisy.